# Lab 16. Automated Forecasting and Prophet-Type Additive Models

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-16-automated-forecasting-prophet-lab.ipynb)

This lab is designed for independent study. It explains the mathematical and modeling ideas before the programming begins, and each code block is followed by interpretation questions.

## What you will learn

By the end of this lab, you should be able to:

1. Explain the idea of an automated forecasting workflow.
2. Build a Prophet-type additive forecasting model from scratch using Python.
3. Represent trend, seasonality, changepoints, and holiday effects as regression features.
4. Compare the additive model with a simple seasonal naive baseline.
5. Use rolling-origin validation to tune model complexity.
6. Diagnose residuals and identify when an automated model is not enough.

## Connection with the chapter

Chapter 16 studies automated forecasting and Prophet-type additive models. The central modeling idea is

$$
y(t) = g(t) + s(t) + h(t) + e_t,
$$

where $g(t)$ is trend, $s(t)$ is seasonality, $h(t)$ represents holiday or event effects, and $e_t$ is unexplained error.

In this lab we do **not** require the external `prophet` package. Instead, we build a transparent Prophet-like model using ordinary numerical tools. This makes the mathematics visible and keeps the notebook reliable in Google Colab.

## Software used

This notebook uses only common Python packages:

- `numpy`
- `pandas`
- `matplotlib`

No special forecasting library is required.

## 1. Background: What does automated forecasting automate?

In earlier chapters, we built models by hand:

- choose a transformation,
- inspect plots,
- choose a model,
- estimate parameters,
- check residuals,
- compare forecasts.

Automated forecasting tries to make this workflow repeatable for many time series. A useful automated model should provide:

1. a default baseline,
2. flexible trend behavior,
3. seasonal components,
4. event or holiday effects,
5. model diagnostics,
6. out-of-sample validation.

Prophet-style models are popular because many business and scientific time series can be approximated by an interpretable decomposition:

$$
y_t = 	ext{trend}_t + 	ext{seasonality}_t + 	ext{holiday effect}_t + 	ext{noise}_t.
$$

This is not the same as ARIMA. ARIMA models direct serial dependence through lagged values and lagged errors. Prophet-type additive models are closer to regression on time, seasonal basis functions, and event indicators.

## 2. Setup

Run the next cell first. We set a random seed so that your plots match the examples.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

print("Packages loaded successfully.")

## 3. Simulate a time series with trend, seasonality, changepoints, and holidays

To understand the model, we first create data where we know the true structure.

The simulated series has:

1. **Piecewise linear trend**: the growth rate changes at two changepoints.
2. **Weekly seasonality**: weekends behave differently from weekdays.
3. **Yearly seasonality**: slow annual variation.
4. **Holiday spikes**: special event days create temporary jumps.
5. **Noise and a few outliers**: real data are never perfectly clean.

This is the kind of structure Prophet-type models are designed to capture.

In [ ]:
def simulate_additive_series(n_days=3 * 365, seed=7339):
    rng_local = np.random.default_rng(seed)
    dates = pd.date_range("2020-01-01", periods=n_days, freq="D")
    t = np.arange(n_days, dtype=float)

    # Piecewise linear trend with two changepoints.
    cp1 = int(0.35 * n_days)
    cp2 = int(0.70 * n_days)
    trend = 80 + 0.035 * t
    trend += 0.060 * np.maximum(t - cp1, 0)
    trend += -0.075 * np.maximum(t - cp2, 0)

    # Weekly and yearly seasonality.
    weekly = 6 * np.sin(2 * np.pi * t / 7) + 2 * np.cos(2 * np.pi * t / 7)
    yearly = 9 * np.sin(2 * np.pi * t / 365.25) + 4 * np.cos(2 * np.pi * t / 365.25)

    # Holiday/event effects.
    holiday = np.zeros(n_days)
    for year in [2020, 2021, 2022]:
        for day in ["01-01", "07-04", "11-26", "12-25"]:
            date = pd.Timestamp(f"{year}-{day}")
            idx = np.where(dates == date)[0]
            if len(idx) > 0:
                holiday[idx[0]] += 22
                if idx[0] + 1 < n_days:
                    holiday[idx[0] + 1] += 10

    noise = rng_local.normal(0, 4, size=n_days)
    y = trend + weekly + yearly + holiday + noise

    # Add a few outliers.
    outlier_idx = rng_local.choice(n_days, size=8, replace=False)
    y[outlier_idx] += rng_local.normal(30, 8, size=len(outlier_idx))

    df = pd.DataFrame({
        "ds": dates,
        "t": t,
        "y": y,
        "trend_true": trend,
        "weekly_true": weekly,
        "yearly_true": yearly,
        "holiday_true": holiday,
        "is_holiday": (holiday > 0).astype(int),
        "is_outlier": 0,
    })
    df.loc[outlier_idx, "is_outlier"] = 1
    return df

series = simulate_additive_series()
series.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(series["ds"], series["y"], linewidth=1)
ax.scatter(
    series.loc[series["is_holiday"] == 1, "ds"],
    series.loc[series["is_holiday"] == 1, "y"],
    s=20,
    label="holiday/event days",
)
ax.set_title("Simulated daily series")
ax.set_xlabel("date")
ax.set_ylabel("value")
ax.legend()
plt.show()

print(series[["y", "is_holiday", "is_outlier"]].describe())

### Checkpoint 1

Look at the plot before modeling.

1. Can you see trend changes?
2. Can you see weekly or yearly seasonality directly?
3. Why might a simple straight-line trend be insufficient?
4. Why is it dangerous to evaluate a forecasting model by randomly shuffling the rows?

## 4. Chronological train/test split

For time series, the test set must occur **after** the training set. Otherwise the model may use future information.

We use the first part of the time series for training and the last 180 days for testing.

In [ ]:
test_size = 180
train = series.iloc[:-test_size].copy()
test = series.iloc[-test_size:].copy()

print("Training period:", train["ds"].min().date(), "to", train["ds"].max().date())
print("Test period:    ", test["ds"].min().date(), "to", test["ds"].max().date())
print("Training observations:", len(train))
print("Test observations:", len(test))

In [ ]:
fig, ax = plt.subplots()
ax.plot(train["ds"], train["y"], label="training")
ax.plot(test["ds"], test["y"], label="test")
ax.axvline(test["ds"].iloc[0], linestyle="--", color="black", linewidth=1)
ax.set_title("Chronological train/test split")
ax.set_xlabel("date")
ax.set_ylabel("value")
ax.legend()
plt.show()

## 5. A seasonal naive baseline

A serious forecasting workflow starts with a baseline. For daily data with weekly seasonality, the seasonal naive forecast is

$$
\widehat{y}_{t+h} = y_{t+h-7}.
$$

This means: use the value from the same day of the previous week.

A complicated model is useful only if it improves on strong simple baselines.

In [ ]:
def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100)

# Seasonal naive: for each test date, use the observed value 7 days earlier.
y_all = series.set_index("ds")["y"]
seasonal_naive_pred = []
for date in test["ds"]:
    seasonal_naive_pred.append(y_all.loc[date - pd.Timedelta(days=7)])

seasonal_naive_pred = np.array(seasonal_naive_pred)

baseline_scores = pd.DataFrame({
    "model": ["seasonal naive"],
    "MAE": [mae(test["y"], seasonal_naive_pred)],
    "RMSE": [rmse(test["y"], seasonal_naive_pred)],
    "MAPE_percent": [mape(test["y"], seasonal_naive_pred)],
})
baseline_scores

In [ ]:
fig, ax = plt.subplots()
ax.plot(test["ds"], test["y"], label="actual")
ax.plot(test["ds"], seasonal_naive_pred, label="seasonal naive forecast")
ax.set_title("Seasonal naive baseline on test period")
ax.set_xlabel("date")
ax.set_ylabel("value")
ax.legend()
plt.show()

### Checkpoint 2

1. Does the baseline capture weekly seasonality?
2. Does the baseline adapt to trend changes?
3. What kinds of changes would make this baseline fail?

## 6. Build Prophet-type features by hand

We now build an interpretable additive regression model.

### 6.1 Trend with changepoints

A piecewise linear trend can be represented using hinge functions:

$$
g(t) = b_0 + b_1 t + d_1(t-s_1)_+ + d_2(t-s_2)_+ + \cdots + d_m(t-s_m)_+,
$$

where

$$
(t-s)_+ = \max(t-s,0).
$$

Before a changepoint $s$, the hinge term is zero. After $s$, it changes the slope.

### 6.2 Fourier seasonality

A seasonal component with period $P$ can be represented using Fourier features:

$$
s(t) = \sum_{k=1}^{K} a_k \cos(2\pi k t/P) + b_k \sin(2\pi k t/P).
$$

Larger $K$ allows more flexible seasonality, but also increases overfitting risk.

### 6.3 Holiday effects

Holiday effects are represented by indicator variables:

$$
h(t) = c \cdot I(t 	ext{ is a holiday}).
$$

In [ ]:
def make_changepoints(t_train, n_changepoints=8, changepoint_range=0.8):
    # Choose equally spaced candidate changepoints inside the training period.
    t_train = np.asarray(t_train, dtype=float)
    start = t_train.min()
    end = t_train.min() + changepoint_range * (t_train.max() - t_train.min())
    if n_changepoints <= 0:
        return np.array([])
    return np.linspace(start, end, n_changepoints + 2)[1:-1]


def fourier_features(t, period, order, prefix):
    # Create sine and cosine Fourier features.
    t = np.asarray(t, dtype=float)
    features = {}
    for k in range(1, order + 1):
        features[f"{prefix}_sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        features[f"{prefix}_cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(features)


def make_design_frame(df, changepoints, weekly_order=3, yearly_order=5):
    # Create an interpretable design matrix for additive forecasting.
    t = np.asarray(df["t"], dtype=float)
    X = pd.DataFrame({
        "intercept": np.ones(len(df)),
        "t_scaled": t / 365.25,
        "is_holiday": df["is_holiday"].to_numpy(dtype=float),
    })

    for j, cp in enumerate(changepoints, start=1):
        X[f"cp_{j}"] = np.maximum(t - cp, 0) / 365.25

    X = pd.concat(
        [
            X,
            fourier_features(t, period=7, order=weekly_order, prefix="weekly"),
            fourier_features(t, period=365.25, order=yearly_order, prefix="yearly"),
        ],
        axis=1,
    )
    return X

changepoints = make_changepoints(train["t"], n_changepoints=8, changepoint_range=0.8)
X_train = make_design_frame(train, changepoints, weekly_order=3, yearly_order=5)
X_test = make_design_frame(test, changepoints, weekly_order=3, yearly_order=5)

print("Design matrix shape:", X_train.shape)
print("First 12 columns:")
print(list(X_train.columns[:12]))

## 7. Fit the additive model using ridge regression

We fit the model

$$
y = Xb + e.
$$

The least-squares solution minimizes

$$
\sum_t (y_t - x_t^T b)^2.
$$

A ridge version adds a penalty on coefficients:

$$
\sum_t (y_t - x_t^T b)^2 + \lambda \sum_j b_j^2.
$$

The penalty makes the fit more stable when many changepoint and Fourier features are included.

Important: we do **not** penalize the intercept.

In [ ]:
def fit_ridge(X, y, lam=1.0, penalize_intercept=False):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    p = X.shape[1]
    penalty = lam * np.eye(p)
    if not penalize_intercept:
        penalty[0, 0] = 0.0
    beta = np.linalg.solve(X.T @ X + penalty, X.T @ y)
    return beta


def predict_linear(X, beta):
    return np.asarray(X, dtype=float) @ beta

lam = 10.0
beta = fit_ridge(X_train, train["y"], lam=lam)
train_pred = predict_linear(X_train, beta)
test_pred = predict_linear(X_test, beta)

model_scores = pd.DataFrame({
    "model": ["seasonal naive", "additive ridge"],
    "MAE": [mae(test["y"], seasonal_naive_pred), mae(test["y"], test_pred)],
    "RMSE": [rmse(test["y"], seasonal_naive_pred), rmse(test["y"], test_pred)],
    "MAPE_percent": [mape(test["y"], seasonal_naive_pred), mape(test["y"], test_pred)],
})
model_scores

In [ ]:
fig, ax = plt.subplots()
ax.plot(train["ds"], train["y"], label="train actual", linewidth=1)
ax.plot(train["ds"], train_pred, label="train fitted", linewidth=1)
ax.plot(test["ds"], test["y"], label="test actual", linewidth=1)
ax.plot(test["ds"], test_pred, label="test forecast", linewidth=2)
ax.axvline(test["ds"].iloc[0], linestyle="--", color="black", linewidth=1)
ax.set_title("Prophet-type additive regression forecast")
ax.set_xlabel("date")
ax.set_ylabel("value")
ax.legend()
plt.show()

### Checkpoint 3

1. Did the additive model beat the seasonal naive baseline?
2. Does the test forecast follow the level of the series?
3. Does the model appear too smooth, too flexible, or reasonable?
4. Which components do you expect to matter most: trend, weekly seasonality, yearly seasonality, or holidays?

## 8. Decompose the fitted forecast into components

One advantage of additive models is interpretability. We can separate fitted values into approximate components.

Because our model is linear in the features, we can group columns into trend, weekly, yearly, and holiday groups.

In [ ]:
feature_names = list(X_train.columns)
beta_series = pd.Series(beta, index=feature_names)

trend_cols = [c for c in feature_names if c in ["intercept", "t_scaled"] or c.startswith("cp_")]
holiday_cols = ["is_holiday"]
weekly_cols = [c for c in feature_names if c.startswith("weekly_")]
yearly_cols = [c for c in feature_names if c.startswith("yearly_")]

X_all = make_design_frame(series, changepoints, weekly_order=3, yearly_order=5)

def component_prediction(X, cols):
    return np.asarray(X[cols]) @ beta_series[cols].to_numpy()

components = pd.DataFrame({
    "ds": series["ds"],
    "trend_fit": component_prediction(X_all, trend_cols),
    "holiday_fit": component_prediction(X_all, holiday_cols),
    "weekly_fit": component_prediction(X_all, weekly_cols),
    "yearly_fit": component_prediction(X_all, yearly_cols),
})
components.head()

In [ ]:
for col, title in [
    ("trend_fit", "Fitted trend component"),
    ("weekly_fit", "Fitted weekly seasonal component"),
    ("yearly_fit", "Fitted yearly seasonal component"),
    ("holiday_fit", "Fitted holiday/event component"),
]:
    fig, ax = plt.subplots()
    ax.plot(components["ds"], components[col])
    ax.set_title(title)
    ax.set_xlabel("date")
    ax.set_ylabel("component value")
    plt.show()

### Checkpoint 4

1. Does the trend component show slope changes near the simulated changepoints?
2. Is the weekly component periodic with period 7?
3. Is the holiday component usually zero except on event days?
4. Why might component plots be helpful when explaining forecasts to nontechnical audiences?

## 9. Rolling-origin validation for model selection

We now tune model complexity using time-aware validation.

We compare different choices of:

- number of changepoints,
- weekly Fourier order,
- yearly Fourier order,
- ridge penalty.

We do not randomly shuffle rows. Instead, each validation split trains on the past and validates on the future.

In [ ]:
def evaluate_additive_config(df, cutoffs, horizon, n_changepoints, weekly_order, yearly_order, lam):
    errors = []
    for cutoff in cutoffs:
        train_cv = df.iloc[:cutoff].copy()
        valid_cv = df.iloc[cutoff:cutoff + horizon].copy()

        cps = make_changepoints(train_cv["t"], n_changepoints=n_changepoints, changepoint_range=0.8)
        X_tr = make_design_frame(train_cv, cps, weekly_order=weekly_order, yearly_order=yearly_order)
        X_va = make_design_frame(valid_cv, cps, weekly_order=weekly_order, yearly_order=yearly_order)

        b = fit_ridge(X_tr, train_cv["y"], lam=lam)
        pred = predict_linear(X_va, b)
        errors.append(mae(valid_cv["y"], pred))
    return float(np.mean(errors))

cutoffs = [500, 650, 800]
horizon = 60

configs = []
for ncp in [2, 6, 10]:
    for w_order in [1, 3]:
        for y_order in [3, 5]:
            for lam_value in [1.0, 10.0, 100.0]:
                cv_mae = evaluate_additive_config(
                    series,
                    cutoffs=cutoffs,
                    horizon=horizon,
                    n_changepoints=ncp,
                    weekly_order=w_order,
                    yearly_order=y_order,
                    lam=lam_value,
                )
                configs.append({
                    "n_changepoints": ncp,
                    "weekly_order": w_order,
                    "yearly_order": y_order,
                    "lambda": lam_value,
                    "cv_MAE": cv_mae,
                })

cv_results = pd.DataFrame(configs).sort_values("cv_MAE")
cv_results.head(10)

In [ ]:
best = cv_results.iloc[0].to_dict()
best

In [ ]:
best_cps = make_changepoints(train["t"], n_changepoints=int(best["n_changepoints"]), changepoint_range=0.8)
X_train_best = make_design_frame(
    train,
    best_cps,
    weekly_order=int(best["weekly_order"]),
    yearly_order=int(best["yearly_order"]),
)
X_test_best = make_design_frame(
    test,
    best_cps,
    weekly_order=int(best["weekly_order"]),
    yearly_order=int(best["yearly_order"]),
)

beta_best = fit_ridge(X_train_best, train["y"], lam=float(best["lambda"]))
test_pred_best = predict_linear(X_test_best, beta_best)

pd.DataFrame({
    "model": ["seasonal naive", "initial additive ridge", "CV-selected additive ridge"],
    "MAE": [mae(test["y"], seasonal_naive_pred), mae(test["y"], test_pred), mae(test["y"], test_pred_best)],
    "RMSE": [rmse(test["y"], seasonal_naive_pred), rmse(test["y"], test_pred), rmse(test["y"], test_pred_best)],
    "MAPE_percent": [mape(test["y"], seasonal_naive_pred), mape(test["y"], test_pred), mape(test["y"], test_pred_best)],
})

In [ ]:
fig, ax = plt.subplots()
ax.plot(test["ds"], test["y"], label="actual", linewidth=1)
ax.plot(test["ds"], seasonal_naive_pred, label="seasonal naive", linewidth=1)
ax.plot(test["ds"], test_pred_best, label="CV-selected additive model", linewidth=2)
ax.set_title("Test forecasts after rolling-origin model selection")
ax.set_xlabel("date")
ax.set_ylabel("value")
ax.legend()
plt.show()

### Checkpoint 5

1. Which configuration had the best rolling-origin MAE?
2. Did the CV-selected model improve the final test score?
3. Why might the model with the lowest validation error not always be the most interpretable model?
4. Why should the test set be used only after model selection is complete?

## 10. Residual diagnostics

A good forecast model should leave residuals that are hard to predict from the past.

The residuals are

$$
e_t = y_t - \widehat{y}_t.
$$

We examine residual plots and the sample autocorrelation function.

In [ ]:
train_pred_best = predict_linear(X_train_best, beta_best)
train_resid = train["y"].to_numpy() - train_pred_best

def sample_acf(x, max_lag):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    denom = np.sum(x ** 2)
    vals = [1.0]
    for h in range(1, max_lag + 1):
        vals.append(float(np.sum(x[h:] * x[:-h]) / denom))
    return np.array(vals)

acf_vals = sample_acf(train_resid, max_lag=40)

fig, ax = plt.subplots()
ax.plot(train["ds"], train_resid)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Training residuals")
ax.set_xlabel("date")
ax.set_ylabel("residual")
plt.show()

fig, ax = plt.subplots()
ax.stem(np.arange(len(acf_vals)), acf_vals)
ax.axhline(1.96 / np.sqrt(len(train_resid)), linestyle="--", color="black", linewidth=1)
ax.axhline(-1.96 / np.sqrt(len(train_resid)), linestyle="--", color="black", linewidth=1)
ax.set_title("Sample ACF of training residuals")
ax.set_xlabel("lag")
ax.set_ylabel("ACF")
plt.show()

### Checkpoint 6

1. Do the residuals look centered around zero?
2. Are there visible outliers?
3. Are there significant residual autocorrelations?
4. If residual autocorrelation remains, what kind of model component might be missing?

## 11. Forecast intervals from residual simulation

The additive model above gives point forecasts. A simple way to get approximate intervals is to use empirical residual quantiles.

This is not a full probabilistic model, but it is a useful first approximation.

In [ ]:
q_lower, q_upper = np.quantile(train_resid, [0.025, 0.975])
lo = test_pred_best + q_lower
hi = test_pred_best + q_upper
coverage = np.mean((test["y"].to_numpy() >= lo) & (test["y"].to_numpy() <= hi))

fig, ax = plt.subplots()
ax.plot(test["ds"], test["y"], label="actual")
ax.plot(test["ds"], test_pred_best, label="point forecast")
ax.fill_between(test["ds"], lo, hi, alpha=0.25, label="approx. 95% interval")
ax.set_title(f"Approximate forecast interval; empirical coverage = {coverage:.3f}")
ax.set_xlabel("date")
ax.set_ylabel("value")
ax.legend()
plt.show()

print("Empirical 95% interval coverage on test set:", round(float(coverage), 3))

### Checkpoint 7

1. Is the empirical coverage close to 0.95?
2. Are the intervals too narrow, too wide, or reasonable?
3. Why might residual-based intervals be unreliable if residuals are autocorrelated or heteroskedastic?

## 12. Optional extension: using the external Prophet package

The notebook above built the main ideas from scratch. In a full project, you may also try the external `prophet` package.

The code below is intentionally turned off so that the notebook runs reliably without installing extra packages. In Google Colab, you may set `RUN_OPTIONAL_PROPHET = True` after installing the package.

In [ ]:
RUN_OPTIONAL_PROPHET = False

if RUN_OPTIONAL_PROPHET:
    try:
        from prophet import Prophet
        prophet_train = train[["ds", "y"]].copy()
        m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        m.fit(prophet_train)
        future = m.make_future_dataframe(periods=test_size, freq="D")
        forecast = m.predict(future)
        print(forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail())
    except Exception as exc:
        print("Prophet is not available in this environment.")
        print("In Colab, try: !pip install prophet")
        print("Error message:", exc)
else:
    print("Optional Prophet package example skipped. Set RUN_OPTIONAL_PROPHET = True to try it.")

## 13. Mini-project

Choose one of the following extensions.

### Project A. Holiday effects

Modify the simulation so that holidays have negative effects instead of positive effects. Refit the model and interpret the holiday coefficient.

### Project B. Changepoint flexibility

Compare models with 0, 2, 5, 10, and 20 changepoints. Plot validation MAE versus number of changepoints. Explain the bias-variance tradeoff.

### Project C. Fourier complexity

Compare yearly Fourier orders $K=1,2,3,5,10$. Which value gives the best rolling-origin score? Do component plots remain interpretable?

### Project D. Real data adaptation

Replace the simulated data with a real daily or monthly data set. Use the same workflow:

1. plot the series,
2. build a baseline,
3. fit an additive model,
4. tune by rolling-origin validation,
5. diagnose residuals,
6. write a short conclusion.

## 14. AI-assisted study prompts

Use an AI assistant only after you have run the notebook and inspected the plots yourself. Good prompts are specific and ask for verification, not just answers.

### Prompt 1. Model decomposition

> I fit an additive model with trend, Fourier seasonality, and holiday indicators. Here are my component plots. Explain which component is responsible for each visible pattern, and list two possible signs of overfitting.

### Prompt 2. Leakage audit

> Review this time-series validation workflow. Identify any step that might leak future information into the training process. Explain how to fix it.

### Prompt 3. Residual diagnosis

> The residual ACF from my additive forecasting model still has significant spikes at lags 1 and 7. What model components or alternative models should I consider next?

### Prompt 4. Executive summary

> Help me write a short forecasting report. Include the baseline, the final model, validation method, test metrics, uncertainty intervals, and limitations.

## 15. Final checklist

Before you finish this lab, make sure you can answer the following questions.

- What is the difference between an additive regression forecast and an ARIMA forecast?
- What does a changepoint feature do mathematically?
- Why are Fourier features useful for seasonality?
- Why do we need a baseline model?
- Why is rolling-origin validation better than random cross-validation for time series?
- What does residual autocorrelation tell us about a fitted forecasting model?
- What can go wrong when automated forecasting is used without diagnostics?

You are now ready to use Prophet-type additive models as interpretable automated forecasting tools, while still checking validation, residuals, and assumptions carefully.